# Stacking and blending
The heart-disease data (303 patients, 13 inputs, `target` 1 = disease). Three base models (random forest, KNN,
gradient boosting) and a logistic regression meta-model: first with scikit-learn's `StackingClassifier`, then
blending and K-fold stacking built by hand, so every step is visible.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 1. The data and the split

In [ ]:
# 303 patients; 'target' is 1 for heart disease, 0 for none
df = pd.read_csv("data/heart.csv")
print(df.shape)
print(df["target"].value_counts())

# inputs and output
X = df.drop(columns=["target"])
y = df["target"]

# 80% to train (242 patients), 20% to test (61 patients)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)
print(X_train.shape, X_test.shape)

## 2. The base models alone

In [ ]:
def base_models(scale_knn=False):
    """The three base models; KNN optionally inside a scaling pipeline."""
    knn = KNeighborsClassifier(n_neighbors=10)
    if scale_knn:
        # KNN measures distances, so put every column on mean 0, std 1 first
        knn = make_pipeline(StandardScaler(), knn)
    return [
        ("rf", RandomForestClassifier(n_estimators=10, random_state=42)),
        ("knn", knn),
        ("gbdt", GradientBoostingClassifier(random_state=42)),
    ]

# each base model trained on the training set and scored on the test set
for name, model in base_models():
    model.fit(X_train, y_train)
    print(name, round(accuracy_score(y_test, model.predict(X_test)), 3))

## 3. StackingClassifier

In [ ]:
# base models -> 10-fold out-of-fold predictions -> logistic regression meta-model
clf = StackingClassifier(
    estimators=base_models(),
    final_estimator=LogisticRegression(max_iter=1000),
    cv=10,
)
clf.fit(X_train, y_train)
print("stacking accuracy:", round(clf.score(X_test, y_test), 3))

# the meta-model's weights: how much it trusts each base model's probability
print(dict(zip(["rf", "knn", "gbdt"], clf.final_estimator_.coef_[0].round(2))))

In [ ]:
# what the meta-model sees: one column per base model (probability of class 1)
print(clf.transform(X_test.iloc[:5]).round(2))

## 4. Scaling KNN, passthrough and soft voting

In [ ]:
results = {}
for name, model in base_models(scale_knn=True):
    results[name] = accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))

# soft voting: average the three probabilities, no meta-model
vote = VotingClassifier(base_models(scale_knn=True), voting="soft").fit(X_train, y_train)
results["soft voting"] = vote.score(X_test, y_test)

# stacking with a scaled KNN
stack = StackingClassifier(base_models(scale_knn=True),
                           final_estimator=LogisticRegression(max_iter=1000), cv=10)
results["stacking"] = stack.fit(X_train, y_train).score(X_test, y_test)

# passthrough=True: the meta-model also gets the 13 original columns (scaled for logistic regression)
stack_pt = StackingClassifier(base_models(scale_knn=True),
                              final_estimator=make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                              cv=10, passthrough=True)
results["stacking, passthrough"] = stack_pt.fit(X_train, y_train).score(X_test, y_test)
print(stack_pt.transform(X_test.iloc[:1]).shape, "meta-model inputs per row")

for k, v in results.items():
    print(f"{k:22s} {v:.3f}")

## 5. Blending by hand (hold-out)

In [ ]:
# split the training set again: base models learn on 'fit', the meta-model on 'val'
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(len(X_fit), "rows for the base models,", len(X_val), "for the meta-model")

models = [m for _, m in base_models(scale_knn=True)]
for m in models:
    m.fit(X_fit, y_fit)

# new dataset: one column per base model, predicted on rows the base models never saw
val_preds = np.column_stack([m.predict_proba(X_val)[:, 1] for m in models])
meta = LogisticRegression().fit(val_preds, y_val)

# prediction: base models first, then the meta-model on their outputs
test_preds = np.column_stack([m.predict_proba(X_test)[:, 1] for m in models])
print("blending accuracy:", round(accuracy_score(y_test, meta.predict(test_preds)), 3))

## 6. K-fold stacking by hand (out-of-fold predictions)

In [ ]:
kf = KFold(n_splits=4, shuffle=True, random_state=42)

# step 1: for each base model, 4 fits; each fold is predicted by the model trained on the other 3
oof = np.column_stack([
    cross_val_predict(m, X_train, y_train, cv=kf, method="predict_proba")[:, 1]
    for _, m in base_models(scale_knn=True)
])
print(oof.shape, "= every training row, one column per base model")

# step 2: the meta-model learns from the out-of-fold predictions
meta = LogisticRegression().fit(oof, y_train)

# step 3: retrain every base model on the whole training set
final = [m.fit(X_train, y_train) for _, m in base_models(scale_knn=True)]

# prediction on the test set
test_preds = np.column_stack([m.predict_proba(X_test)[:, 1] for m in final])
print("K-fold stacking accuracy:", round(accuracy_score(y_test, meta.predict(test_preds)), 3))

## 7. Why not predict on the training data?

In [ ]:
# the naive recipe: base models predict the very rows they were trained on
naive = np.column_stack([m.predict_proba(X_train)[:, 1] for m in final])
print("training-row predictions, mean |p - y|:")
print("  in-sample   ", np.abs(naive - y_train.values[:, None]).mean(axis=0).round(3))
print("  out-of-fold ", np.abs(oof - y_train.values[:, None]).mean(axis=0).round(3))
meta_naive = LogisticRegression().fit(naive, y_train)
print("naive meta weights      ", meta_naive.coef_[0].round(2))
print("out-of-fold meta weights", meta.coef_[0].round(2))
print("naive stacking accuracy:", round(accuracy_score(y_test, meta_naive.predict(test_preds)), 3))